# SwissFlakes Compliance Analysis

Deep-dive into regulatory compliance data:
- **FINMA** - Financial Market Supervisory Authority reporting
- **BAZG** - Swiss Customs Administration declarations
- **GwG** - Anti-Money Laundering Act flags

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
print(f"Connected: {session.get_current_user()} @ {session.get_current_database()}")

## Load Compliance Transaction Report

In [ ]:
try:
    df = session.sql("SELECT * FROM SFG_ENTERPRISE.MART_COMPLIANCE.TRANSACTION_REPORT").to_pandas()
    print(f"Total transactions: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    df.head()
except Exception as e:
    print(f"Data not available: {e}")
    print("Run: EXECUTE DBT PROJECT SFG_ENTERPRISE.DCM.DBT_COMPLIANCE ARGS = 'run';")
    df = pd.DataFrame()

## AML Analysis (GwG)

Identify transactions flagged for anti-money laundering review.

In [ ]:
if not df.empty and "REQUIRES_AML_CHECK" in df.columns:
    aml = df[df["REQUIRES_AML_CHECK"] == True]
    print(f"AML-flagged transactions: {len(aml)} / {len(df)} ({len(aml)/len(df)*100:.1f}%)")
    print(f"\nTotal amount flagged: CHF {aml["AMOUNT_CHF"].sum():,.2f}" if "AMOUNT_CHF" in aml.columns else "")
    print(f"\nPayment methods:")
    if "PAYMENT_METHOD" in aml.columns:
        print(aml["PAYMENT_METHOD"].value_counts().to_string())
    aml.head(20)
else:
    print("No AML data available")

## Customs Declarations (BAZG)

Shipments requiring customs clearance.

In [ ]:
if not df.empty and "REQUIRES_CUSTOMS" in df.columns:
    customs = df[df["REQUIRES_CUSTOMS"] == True]
    print(f"Customs-required: {len(customs)} / {len(df)}")
    if "CUSTOMS_DECLARATION_ID" in customs.columns:
        declared = customs["CUSTOMS_DECLARATION_ID"].notna().sum()
        print(f"With declaration ID: {declared}")
        print(f"Missing declaration: {len(customs) - declared}")
    customs.head(20)
else:
    print("No customs data available")

## International Transaction Overview

In [ ]:
if not df.empty and "IS_INTERNATIONAL" in df.columns:
    intl = df[df["IS_INTERNATIONAL"] == True]
    domestic = df[df["IS_INTERNATIONAL"] == False]
    print(f"International: {len(intl)}  |  Domestic: {len(domestic)}")
    if "AMOUNT_CHF" in df.columns:
        print(f"International volume: CHF {intl["AMOUNT_CHF"].sum():,.2f}")
        print(f"Domestic volume:      CHF {domestic["AMOUNT_CHF"].sum():,.2f}")
else:
    print("No international flag data available")

## Risk Summary

Combined risk indicators per company.

In [ ]:
if not df.empty and "COMPANY_NAME" in df.columns:
    risk = df.groupby("COMPANY_NAME").agg(
        total_txn=("PAYMENT_ID", "count"),
        total_chf=("AMOUNT_CHF", "sum"),
        aml_flags=("REQUIRES_AML_CHECK", "sum"),
        customs_req=("REQUIRES_CUSTOMS", "sum"),
        international=("IS_INTERNATIONAL", "sum"),
    ).sort_values("aml_flags", ascending=False)
    print("Top companies by AML flags:")
    risk.head(20)
else:
    print("Company data not available")

## Semantic View: Compliance Analytics

Test the compliance semantic view.

In [ ]:
try:
    desc = session.sql("DESCRIBE SEMANTIC VIEW SFG_ENTERPRISE.MART_COMPLIANCE.COMPLIANCE_ANALYTICS").to_pandas()
    desc
except Exception as e:
    print(f"Semantic view not deployed: {e}")